# 🚀 Модуль 12. Урок 3. Автономный агент мониторинга аномалий и сбора данных

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_12_ai_agents/lesson_03_automated_data_collection_agent.ipynb)

## 📌 Бизнес-контекст и методология
Автономный агент запускается по расписанию (cron), опрашивает базу данных, сравнивает метрики с доверительными интервалами и отправляет алерт в Slack/Telegram при выходе за пределы нормы.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np

# Логика детекции аномалий в автономном агенте
def anomaly_detector_agent(current_value, history_values):
    mean_hist = np.mean(history_values)
    std_hist = np.std(history_values)
    z_score = (current_value - mean_hist) / std_hist if std_hist > 0 else 0
    
    is_anomaly = abs(z_score) > 2.5
    severity = "CRITICAL" if abs(z_score) > 3.5 else ("WARNING" if is_anomaly else "NORMAL")
    return {"is_anomaly": is_anomaly, "z_score": round(z_score, 2), "severity": severity}

history = [100, 102, 98, 105, 99, 101, 103]
alert = anomaly_detector_agent(145, history) # Внезапный скачок
print(f"Результат работы агента мониторинга: {alert}")


Результат работы агента мониторинга: {'is_anomaly': np.True_, 'z_score': np.float64(19.65), 'severity': 'CRITICAL'}


### 🎯 Практическое задание 1: Детекция критической аномалии
Убедитесь, что для значения 145 агент выставил флаг `is_anomaly = True`.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert alert["is_anomaly"] == True
assert alert["severity"] in ["WARNING", "CRITICAL"]
print("✅ Тест успешно пройден: аномалия успешно зафиксирована агентом!")


✅ Тест успешно пройден: аномалия успешно зафиксирована агентом!


### 💡 Борьба с Alert Fatigue
Не отправляйте алерты по каждому шуму. Группируйте уведомления и настраивайте скользящие окна сглаживания.